<a href="https://colab.research.google.com/github/group-geopulse/GeoPulse/blob/main/project_trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Using GDELT API via gdeltdoc

In [ ]:
%pip install gdeltdoc

In [ ]:
import gdeltdoc
from gdeltdoc import Filters
from datetime import datetime, timedelta

yesterday = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
day_before_yesterday = (datetime.now() - timedelta(days=2)).strftime('%Y-%m-%d')

f = Filters(
    start_date = day_before_yesterday,
    end_date = yesterday,
    domain = ['bloomberg.com', 'reuters.com']
    )

gd = gdeltdoc.GdeltDoc()

# Search for articles matching the filters
articles = gd.article_search(f)

In [ ]:
articles.groupby(['domain']).count()

## Notes
- Does not return Tone or any other sentiment analysis score
- Does not appear to support Financial Times news - 'ft.com'
- Can't filter by language but seems to bias towards english anyways
- All news seems to be only from the US



## GDELT via BigQuery

In [ ]:
import pandas as pd

from google.cloud import bigquery
from google.colab import auth
from google.colab import drive

auth.authenticate_user()
drive.mount('drive')
client = bigquery.Client(project='gdelt-bq-4514')

In [ ]:
### Last line the query has been commented out as this can be done via the dataframe
### May need to uncomment to limit the search as attempting to retrieve a very large result will fail

##### Date, Source, Headline, Link, Snippet, Tone, Positive Score, Negative Score, Polarity

retrieval_query = (f'''
    SELECT
      Date,
      SourceCommonName as Source,
      REGEXP_EXTRACT(Extras, r'<PAGE_TITLE>(.*?)</PAGE_TITLE>') as Headline,
      DocumentIdentifier as Link,
      V2Tone
    FROM
      `gdelt-bq.gdeltv2.gkg`
    WHERE
      Date > 20100000000000 AND
      Date < 20200000000000 AND
      SourceCollectionIdentifier = 1 AND
      Extras LIKE '%<PAGE_TITLE>_%</PAGE_TITLE>%' AND
      TranslationInfo IS NULL AND
      SourceCommonName IN ('bloomberg.com', 'reuters.com', 'ft.com')
      -- LOWER(REGEXP_EXTRACT(Extras, r'<PAGE_TITLE>(.*?)</PAGE_TITLE>')) LIKE '%oil%'
  ''')


results = client.query(retrieval_query).to_dataframe()

### Export dataframe as CSV
Saved in personal Google Drive under Folder 'group-project'


In [ ]:
results.to_csv('gdelt_news_5_years.csv', index=False)
!cp gdelt_news_5_years.csv 'drive/My Drive/group-project'

### Load CSV from Google Drive

In [ ]:
results = pd.read_csv('drive/My Drive/group-project/gdelt_news_5_years.csv')

In [ ]:
### Extract data from column 'V2Tone' into relevant columns
results[['Tone', 'Positive Score', 'Negative Score', 'Polarity', 'Excess']] = results['V2Tone'].str.split(pat=',', n=4, expand=True)

### Drop column 'V2Tone' and 'Excess'
results = results.drop(columns=['V2Tone', 'Excess'])

In [ ]:
display(results)

In [ ]:
### Checking if there are duplicate Headline in the dataset
results['Headline'].value_counts().loc[lambda x: x > 1]

In [ ]:
### Drop duplicate values
filtered_results = results[~results['Headline'].isin(results['Headline'].value_counts().loc[lambda x: x > 1].index)]

In [ ]:
### Extract rows with Healine that contain given keywords

# keywords = ['oil', 'war', 'political', 'geopolitical']
keywords = ['tensions', 'crude', 'oil prices', 'oil supply', 'disruption', 'brent', 'sanctions', 'embargo', 'opec', 'middle east', 'russia', 'ukraine', 'petroleum', 'fuel', 'energy', 'climate', 'global warming']

### Adding regex pattern matching to each keyword string
for i in range(len(keywords)):
  keywords[i] = f'''( |[^a-z]|[^A-Z]){keywords[i]}( |[^a-z]|[^A-Z])'''

# filtered_results = filtered_results[filtered_results['Headline'].str.contains('|'.join(keywords), case=False, regex=True)]

for i in range(len(keywords)):
  string = f'''LOWER(REGEXP_EXTRACT(Extras, r'<PAGE_TITLE>(.*?)</PAGE_TITLE>')) LIKE '{keywords[i]}' OR'''
  print(string)

In [ ]:
display(filtered_results.info())

### Save & Load CSV from Google Drive

In [ ]:
filtered_results.to_csv('filtered_news_5_years.csv', index=False)
!cp filtered_news_5_years.csv 'drive/My Drive/group-project'

In [ ]:
filtered_results = pd.read_csv('drive/My Drive/group-project/filtered_news_5_years.csv')

In [ ]:
display(filtered_results)